In [1]:
import pandas as pd
import numpy as np

# Load all the necessary tables
products = pd.read_csv('../data/products.csv')
customers = pd.read_csv('../data/customers.csv')
promotions = pd.read_csv('../data/promotions.csv')
geography = pd.read_csv('../data/geography.csv')
orders = pd.read_csv('../data/orders.csv')
order_items = pd.read_csv('../data/order_items.csv')
payments = pd.read_csv('../data/payments.csv')
returns = pd.read_csv('../data/returns.csv')
web_traffic = pd.read_csv('../data/web_traffic.csv')

# Ensure date columns are actually parsed as datetime objects for calculations
orders['order_date'] = pd.to_datetime(orders['order_date'])

/tmp/ipykernel_515979/2557676701.py:10: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv('../data/order_items.csv')


In [2]:
print("--- Q1: Median Inter-Order Gap ---")

# Sort orders chronologically per customer
orders_sorted = orders.sort_values(['customer_id', 'order_date'])

# Calculate the difference in days between current and previous order for each customer
orders_sorted['days_since_last'] = orders_sorted.groupby('customer_id')['order_date'].diff().dt.days

# Drop NaNs (which represent the first order for every customer) and find median
median_gap = orders_sorted['days_since_last'].dropna().median()

print(f"Answer: {median_gap} days")

--- Q1: Median Inter-Order Gap ---
Answer: 144.0 days


In [3]:
print("\n--- Q2: Highest Gross Profit Margin Segment ---")

# Calculate margin for each product
products['margin'] = (products['price'] - products['cogs']) / products['price']

# Group by segment, calculate mean, and sort descending
segment_margins = products.groupby('segment')['margin'].mean().sort_values(ascending=False)

print(segment_margins)
print(f"Answer: {segment_margins.idxmax()}")


--- Q2: Highest Gross Profit Margin Segment ---
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: margin, dtype: float64
Answer: Standard


In [4]:
print("\n--- Q3: Most Common Return Reason (Streetwear) ---")

# Join returns with products to get the category
returns_with_category = returns.merge(products[['product_id', 'category']], on='product_id')

# Filter for Streetwear and find the most frequent reason
streetwear_returns = returns_with_category[returns_with_category['category'] == 'Streetwear']
top_reason = streetwear_returns['return_reason'].mode()[0]

print(f"Answer: {top_reason}")


--- Q3: Most Common Return Reason (Streetwear) ---
Answer: wrong_size


In [5]:
print("\n--- Q4: Lowest Bounce Rate Traffic Source ---")

# Group by traffic source and find the mean bounce rate
bounce_rates = web_traffic.groupby('traffic_source')['bounce_rate'].mean().sort_values()

print(bounce_rates)
print(f"Answer: {bounce_rates.idxmin()}")


--- Q4: Lowest Bounce Rate Traffic Source ---
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64
Answer: email_campaign


In [6]:
print("\n--- Q5: Percentage of Promoted Order Items ---")

# Count rows where promo_id is not null, divide by total rows
promo_count = order_items['promo_id'].notna().sum()
total_items = len(order_items)
promo_percentage = (promo_count / total_items) * 100

print(f"Answer: {promo_percentage:.0f}%")


--- Q5: Percentage of Promoted Order Items ---
Answer: 39%


In [7]:
print("\n--- Q6: Highest Average Orders per Customer by Age Group ---")

# Filter out customers with null age groups
valid_customers = customers[customers['age_group'].notna()]

# Merge to get order data for these valid customers
cust_orders = valid_customers.merge(orders, on='customer_id', how='left')

# Count total unique orders per age group
total_orders_per_group = cust_orders.groupby('age_group')['order_id'].nunique()

# Count total unique customers per age group
total_customers_per_group = valid_customers.groupby('age_group')['customer_id'].nunique()

# Calculate the ratio
avg_orders = (total_orders_per_group / total_customers_per_group).sort_values(ascending=False)

print(avg_orders)
print(f"Answer: {avg_orders.idxmax()}")


--- Q6: Highest Average Orders per Customer by Age Group ---
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
dtype: float64
Answer: 55+


In [8]:
print("\n--- Q7: Region with Highest Total Revenue ---")

# Join order_items to orders (to get ZIP code)
transactions = order_items.merge(orders[['order_id', 'zip']], on='order_id')

# Calculate actual revenue per item row
transactions['item_revenue'] = (transactions['quantity'] * transactions['unit_price']) - transactions['discount_amount'].fillna(0)

# Join with geography to get the region
transactions_geo = transactions.merge(geography[['zip', 'region']], on='zip')

# Group by region and sum the revenue
regional_revenue = transactions_geo.groupby('region')['item_revenue'].sum().sort_values(ascending=False)

print(regional_revenue)
print(f"Answer: {regional_revenue.idxmax()}")


--- Q7: Region with Highest Total Revenue ---
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: item_revenue, dtype: float64
Answer: East


In [9]:
print("\n--- Q8: Top Payment Method for Cancelled Orders ---")

# Filter for cancelled orders directly in the orders table
# Since payment_method is already in the orders table, we don't need a join
cancelled_orders = orders[orders['order_status'] == 'cancelled']

# Find the most frequent payment method
top_payment = cancelled_orders['payment_method'].mode()[0]

print(f"Answer: {top_payment}")


--- Q8: Top Payment Method for Cancelled Orders ---
Answer: credit_card


In [10]:
print("\n--- Q9: Highest Return Rate by Size ---")

# Join both transaction tables to products to get the 'size' column
oi_prod = order_items.merge(products[['product_id', 'size']], on='product_id')
ret_prod = returns.merge(products[['product_id', 'size']], on='product_id')

# Count occurrences of each size in orders and returns
size_ordered_count = oi_prod['size'].value_counts()
size_returned_count = ret_prod['size'].value_counts()

# Calculate the rate (fill NaNs with 0 in case a size was never returned)
return_rates = (size_returned_count / size_ordered_count).fillna(0)

# Filter for only S, M, L, XL as requested by the options
target_sizes = return_rates[['S', 'M', 'L', 'XL']].sort_values(ascending=False)

print(target_sizes)
print(f"Answer: {target_sizes.idxmax()}")


--- Q9: Highest Return Rate by Size ---
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
Name: count, dtype: float64
Answer: S


In [11]:
print("\n--- Q10: Highest Avg Payment Value by Installments ---")

# Group by installments and calculate mean payment value
avg_payment_by_installment = payments.groupby('installments')['payment_value'].mean().sort_values(ascending=False)

print(avg_payment_by_installment)
print(f"Answer: {avg_payment_by_installment.idxmax()} kỳ")


--- Q10: Highest Avg Payment Value by Installments ---
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64
Answer: 6 kỳ
